# graph-vul: CWE-Matching Ablation Workflow on Google Colab

This notebook runs the full function-level CWE-matching ablation workflow on Google Colab with Google Drive persistence.

It is separate from `notebooks/colab_finetune.ipynb` and is designed for **multi-variant** runs with **shared hyperparameters** across selected ablation modes.

## Workflow

For each variant in `ABLATION_MODES`, the notebook will:
1. Train `graph_vul.train_finetune`
2. Export split predictions with retrieval-aware outputs
3. Run tier-1 external evaluation
4. Save variant-specific checkpoints, metrics, and reports to Drive

## Expected Drive Layout

Before running, upload the following to `MyDrive/data/graph-vul/data/`:

```text
MyDrive/
  data/
    graph-vul/
      data/
        func_subset.csv
        func_splits/
          assignments.csv
        partitions/              # only if using PARTITION
          short/
            func_subset.csv
            splits/
              assignments.csv
          other/
            func_subset.csv
            splits/
              assignments.csv
        processed/
          cwe_embeddings/
            <one .pt file per referenced CWE>
          graph_artifacts.tar.gz  # only if USE_GRAPH = True
```

## Outputs

Drive outputs are written under:
- `data/experiments/cwe_matching/<variant>/func_finetune_model.ckpt`
- `data/experiments/cwe_matching/<variant>/func_finetune_metrics.json`
- `data/analysis/func_finetune_predictions/<variant_lower>/...`
- `data/analysis/external_eval_tier1/<variant_lower>/...`


In [ ]:
import subprocess

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError("No GPU detected. Go to Runtime > Change runtime type > GPU.")
print(gpu_info.stdout)


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/data/graph-vul'
DRIVE_DATA = f'{DRIVE_PROJECT}/data'
DRIVE_HF_CACHE = f'{DRIVE_PROJECT}/hf_cache'
DRIVE_GRAPH_ARCHIVE = f'{DRIVE_DATA}/processed/graph_artifacts.tar.gz'
LOCAL_GRAPH_ROOT = '/content/graph_artifacts_cache'
LOCAL_GRAPH_DIR = f'{LOCAL_GRAPH_ROOT}/graph_artifacts'


In [ ]:
!mkdir -p {DRIVE_DATA}
!mkdir -p {DRIVE_HF_CACHE}
!mkdir -p {DRIVE_DATA}/experiments/cwe_matching
!mkdir -p {DRIVE_DATA}/analysis/func_finetune_predictions
!mkdir -p {DRIVE_DATA}/analysis/external_eval_tier1


## Multi-Variant Configuration

Set shared hyperparameters once. The notebook will run the full workflow for every variant in `ABLATION_MODES`.


In [ ]:
ABLATION_MODES = ['E0', 'E1', 'E2', 'E3', 'E4']
SEEDS = [42, 43, 44]  # multi-seed sweep; set to [42] for single-seed runs
RUN_EXPORT = True
RUN_EXTERNAL_EVAL = True

NUM_EPOCHS = 30
BATCH_SIZE = 8
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
GRAD_ACCUM = 8
USE_GRAPH = True
FUSION_LAYERS = list(range(12))
FUSION_METHOD = 'scalar_gated_residual'
FINAL_GRAPH_CONCAT = True
FREEZE_ENCODER_EPOCHS = 3
UNFREEZE_ENCODER_LR = 5e-6
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
HEAD_DROPOUT = 0.5
FREEZE_ENCODER_FULLY = False
PARTITION = 'other'  # None, 'short', or 'other'
GRAPH_ARCHIVE_NAME = 'graph_artifacts.tar.gz'
DRIVE_GRAPH_ARCHIVE = f'{DRIVE_DATA}/processed/{GRAPH_ARCHIVE_NAME}'

MATCHING_PROJECTION_DIM = 128
MATCHING_NEGATIVES_PER_POSITIVE = 1
MATCHING_LOSS_LAMBDA = 0.1
MATCHING_TRIPLET_MARGIN = 0.2

PRINT_ONLY = False  # set True to inspect resolved paths without running the workflow
SKIP_COMPLETED_VARIANTS = True  # skip a (variant, seed) when its checkpoint already exists
FORCE_RERUN_VARIANTS = []  # e.g. ['E3'] to rerun specific variants even when checkpoint exists


In [ ]:
import csv
from pathlib import Path

data_root = Path(DRIVE_DATA)

try:
    _partition = PARTITION
except NameError:
    _partition = None

if _partition in ('short', 'other'):
    subset_path = data_root / 'partitions' / _partition / 'func_subset.csv'
    assignments_path = data_root / 'partitions' / _partition / 'splits' / 'assignments.csv'
    print(f'Using partition: {_partition}')
else:
    subset_path = data_root / 'func_subset.csv'
    assignments_path = data_root / 'func_splits' / 'assignments.csv'
    print('Using full dataset (no partition)')

unseen_assignments_path = assignments_path.parent / 'unseen_assignments.csv'
embeddings_dir = data_root / 'processed' / 'cwe_embeddings'

required_files = [subset_path, assignments_path, embeddings_dir]
missing = [str(path.relative_to(data_root)) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        f'Missing required paths in {DRIVE_DATA}:\n' + '\n'.join(f'  - {path}' for path in missing)
    )

required_splits = {'train', 'val', 'test'}
with assignments_path.open(newline='') as handle:
    reader = csv.DictReader(handle)
    observed_splits = {row['split'] for row in reader if row.get('split')}

missing_splits = sorted(required_splits - observed_splits)
if missing_splits:
    raise ValueError(
        'assignments.csv is missing required splits:\n' + '\n'.join(f'  - {split}' for split in missing_splits)
    )

required_unseen_splits = {'unseen_val', 'unseen_test'}
unseen_in_main = required_unseen_splits.issubset(observed_splits)
if unseen_in_main:
    print('Unseen splits found in assignments.csv (project-disjoint mode)')
elif unseen_assignments_path.exists():
    with unseen_assignments_path.open(newline='') as handle:
        reader = csv.DictReader(handle)
        observed_unseen_splits = {row['split'] for row in reader if row.get('split')}
    missing_unseen = sorted(required_unseen_splits - observed_unseen_splits)
    if missing_unseen:
        raise ValueError(
            'unseen_assignments.csv is missing required splits:\n' + '\n'.join(f'  - {split}' for split in missing_unseen)
        )
    print('Unseen splits found in unseen_assignments.csv (official mode)')
else:
    print('WARNING: No unseen splits found. Training will use train/val/test only.')

with subset_path.open(newline='') as handle:
    reader = csv.DictReader(handle)
    cwe_ids = sorted({
        cwe_id.strip()
        for row in reader
        for cwe_id in [row.get('cwe_id') or '']
        if cwe_id.strip()
    })

missing_embeddings = [cwe_id for cwe_id in cwe_ids if not (embeddings_dir / f'{cwe_id}.pt').exists()]
if missing_embeddings:
    raise FileNotFoundError(
        'Missing CWE embeddings for func_subset.csv entries:\n'
        + '\n'.join(f'  - processed/cwe_embeddings/{cwe_id}.pt' for cwe_id in missing_embeddings)
    )

print(f'Validated dataset root: {data_root}')
print(f'Found {len(cwe_ids)} CWE embedding files referenced by func_subset.csv.')
print(f'Observed primary splits: {sorted(observed_splits)}')


In [ ]:
import os

os.makedirs(DRIVE_HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')


In [ ]:
from google.colab import userdata

github_token = userdata.get('GITHUB_TOKEB_GRAPH_VUL')
if not github_token:
    raise ValueError('Missing GITHUB_TOKEB_GRAPH_VUL in Colab secrets')

!git clone https://{github_token}@github.com/noamdwc/graph-vul.git /content/graph-vul
%pip install -q pytorch-lightning transformers sentencepiece pandas numpy scikit-learn torchmetrics wandb torch-geometric datasets tabulate


In [ ]:
# Validate graph artifacts only when USE_GRAPH is enabled.
from pathlib import Path

data_root = Path(DRIVE_DATA)
try:
    _use_graph = USE_GRAPH
except NameError:
    _use_graph = False

if _use_graph:
    graph_dir = data_root / 'processed' / 'graph_artifacts'
    graph_archive = Path(DRIVE_GRAPH_ARCHIVE)
    if graph_archive.exists():
        print(f'Graph mode enabled: found archive {graph_archive}')
    elif graph_dir.exists():
        node_vocab = graph_dir / 'node_type_vocab.json'
        edge_vocab = graph_dir / 'edge_type_vocab.json'
        missing_graph = []
        if not node_vocab.exists():
            missing_graph.append(str(node_vocab.relative_to(data_root)))
        if not edge_vocab.exists():
            missing_graph.append(str(edge_vocab.relative_to(data_root)))
        if missing_graph:
            raise FileNotFoundError(
                f'USE_GRAPH=True but missing graph artifacts in {DRIVE_DATA}:\n' + '\n'.join(f'  - {p}' for p in missing_graph)
            )
        sample_graph = next(graph_dir.glob('*.pt'), None)
        if sample_graph is None:
            raise FileNotFoundError(f'USE_GRAPH=True but no graph .pt files were found in {graph_dir}')
        print(f'Graph mode enabled: using extracted artifacts in {graph_dir}')
    else:
        raise FileNotFoundError(f'USE_GRAPH=True but neither {graph_dir} nor {graph_archive} exists')
else:
    print('Graph mode disabled (USE_GRAPH=False). Skipping graph validation.')


In [ ]:
import shutil
import subprocess
from pathlib import Path

repo_root = Path('/content/graph-vul')
repo_data = repo_root / 'data'
repo_processed = repo_data / 'processed'
drive_data = Path(DRIVE_DATA)
drive_processed = drive_data / 'processed'
drive_graph_dir = drive_processed / 'graph_artifacts'
drive_graph_archive = Path(DRIVE_GRAPH_ARCHIVE)
local_graph_dir = Path(LOCAL_GRAPH_DIR)

def reset_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.is_dir():
        shutil.rmtree(path)

def symlink_path(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    reset_path(dst)
    dst.symlink_to(src, target_is_directory=src.is_dir())

reset_path(repo_data)
repo_processed.mkdir(parents=True, exist_ok=True)

for relative_path in ['func_subset.csv', 'func_splits', 'partitions']:
    src = drive_data / relative_path
    if src.exists():
        symlink_path(src, repo_data / relative_path)

symlink_path(drive_processed / 'cwe_embeddings', repo_processed / 'cwe_embeddings')

if drive_graph_archive.exists():
    local_graph_dir.parent.mkdir(parents=True, exist_ok=True)
    if not local_graph_dir.exists():
        print(f'Extracting {drive_graph_archive} to {local_graph_dir.parent} ...')
        members = subprocess.run(['tar', '-tzf', str(drive_graph_archive)], capture_output=True, text=True, check=True).stdout.splitlines()
        members = [member.strip() for member in members if member.strip()]
        if not members:
            raise ValueError(f'Archive is empty: {drive_graph_archive}')
        graph_member = next((member for member in members if 'graph_artifacts/' in member), None)
        if graph_member is None:
            raise ValueError(f'Archive does not contain graph_artifacts: {drive_graph_archive}')
        strip_components = Path(graph_member).parts.index('graph_artifacts')
        reset_path(local_graph_dir)
        subprocess.run([
            'tar', '-xzf', str(drive_graph_archive), '-C', str(local_graph_dir.parent), f'--strip-components={strip_components}'
        ], check=True)
        node_vocab = local_graph_dir / 'node_type_vocab.json'
        edge_vocab = local_graph_dir / 'edge_type_vocab.json'
        if not node_vocab.exists() or not edge_vocab.exists():
            raise ValueError('Extracted archive does not contain node_type_vocab.json and edge_type_vocab.json')
    else:
        print(f'Reusing local graph cache at {local_graph_dir}')
    symlink_path(local_graph_dir, repo_processed / 'graph_artifacts')
elif drive_graph_dir.exists():
    symlink_path(drive_graph_dir, repo_processed / 'graph_artifacts')
    print('Graph artifacts will be read directly from Drive (slower for training).')
else:
    print('No graph artifacts found during setup. This is fine if USE_GRAPH stays False.')

assert Path(repo_data, 'func_subset.csv').exists(), 'Dataset wiring failed: func_subset.csv missing'
assert Path(repo_processed, 'cwe_embeddings').exists(), 'Dataset wiring failed: cwe embeddings missing'
print(f'Prepared repo data tree at {repo_data}')
print(f'Primary data source: {drive_data}')
if (repo_processed / 'graph_artifacts').exists():
    print(f'Graph artifacts path: {repo_processed / "graph_artifacts"}')


## Weights & Biases

The training script logs to W&B. Run the next cell to authenticate.
To disable logging, set `WANDB_MODE=offline` before running training.


In [ ]:
import os
from google.colab import userdata

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
!wandb login


In [ ]:
import gc
import json
import logging
import sys
from pathlib import Path

import pandas as pd
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
sys.path.insert(0, '/content/graph-vul')

import graph_vul.config as _cfg

if PARTITION in ('short', 'other'):
    _cfg.FUNC_SUBSET_CSV = _cfg.DATA_DIR / 'partitions' / PARTITION / 'func_subset.csv'
    _cfg.FUNC_SPLITS_DIR = _cfg.DATA_DIR / 'partitions' / PARTITION / 'splits'
    print(f"Config overridden for partition '{PARTITION}':")
    print(f'  FUNC_SUBSET_CSV = {_cfg.FUNC_SUBSET_CSV}')
    print(f'  FUNC_SPLITS_DIR = {_cfg.FUNC_SPLITS_DIR}')

from graph_vul.train_finetune import train
from graph_vul.export_finetune_predictions import export_predictions
from graph_vul.eval_external import evaluate_external

base_experiment_dir = Path(DRIVE_DATA) / 'experiments' / 'cwe_matching'
predictions_base = Path(DRIVE_DATA) / 'analysis' / 'func_finetune_predictions'
external_base = Path(DRIVE_DATA) / 'analysis' / 'external_eval_tier1'
results = []

force_rerun_set = {v.upper() for v in FORCE_RERUN_VARIANTS}

for variant in ABLATION_MODES:
    variant = variant.upper()
    variant_lower = variant.lower()
    force_rerun = variant in force_rerun_set

    for seed in SEEDS:
        run_tag = f'colab-{variant_lower}-seed{seed}' if len(SEEDS) > 1 else f'colab-{variant_lower}'
        seed_subdir = f'seed{seed}' if len(SEEDS) > 1 else ''

        variant_dir = base_experiment_dir / variant_lower / seed_subdir if seed_subdir else base_experiment_dir / variant_lower
        variant_dir.mkdir(parents=True, exist_ok=True)

        checkpoint_path = variant_dir / 'func_finetune_model.ckpt'
        metrics_path = variant_dir / 'func_finetune_metrics.json'
        prediction_dir = (predictions_base / variant_lower / seed_subdir) if seed_subdir else (predictions_base / variant_lower)
        external_dir = (external_base / variant_lower / seed_subdir) if seed_subdir else (external_base / variant_lower)

        export_output_base = prediction_dir.parent
        external_output_base = external_dir.parent

        print('=' * 90)
        print(f'Running variant {variant} (seed={seed})')
        print(f'  checkpoint: {checkpoint_path}')
        print(f'  metrics:    {metrics_path}')
        print(f'  export dir: {prediction_dir}')
        print(f'  external:   {external_dir}')

        if PRINT_ONLY:
            results.append({
                'variant': variant,
                'seed': seed,
                'checkpoint_path': str(checkpoint_path),
                'metrics_path': str(metrics_path),
                'prediction_dir': str(prediction_dir),
                'external_dir': str(external_dir),
                'status': 'print_only',
            })
            continue

        if checkpoint_path.exists() and not force_rerun:
            print(f'WARNING: existing checkpoint found for {variant} seed={seed} at {checkpoint_path}')
            print('  If this checkpoint was created before the checkpoint-serialization fix, export/eval may fail on load.')
            print('  Delete it or add the variant to FORCE_RERUN_VARIANTS to regenerate a safe checkpoint.')
        if SKIP_COMPLETED_VARIANTS and checkpoint_path.exists() and not force_rerun:
            print(f'Skipping {variant} seed={seed}: checkpoint already exists at {checkpoint_path}')
            metrics = {}
            if metrics_path.exists():
                metrics = json.loads(metrics_path.read_text())
            results.append({
                'variant': variant,
                'seed': seed,
                'checkpoint_path': str(checkpoint_path),
                'metrics_path': str(metrics_path),
                'prediction_dir': str(prediction_dir),
                'external_dir': str(external_dir),
                'test_f1': metrics.get('test_f1'),
                'unseen_test_f1': metrics.get('unseen_test_f1'),
                'test_0_64_f1': metrics.get('test_0_64_f1'),
                'unseen_test_0_64_f1': metrics.get('unseen_test_0_64_f1'),
                'optimal_threshold': metrics.get('optimal_threshold'),
                'status': 'skipped_existing',
            })
            continue

        train(
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            encoder_lr=ENCODER_LR,
            head_lr=HEAD_LR,
            grad_accum=GRAD_ACCUM,
            seed=seed,
            use_graph=USE_GRAPH,
            fusion_layers=FUSION_LAYERS,
            fusion_method=FUSION_METHOD,
            final_graph_concat=FINAL_GRAPH_CONCAT,
            checkpoint_path=checkpoint_path,
            metrics_path=metrics_path,
            freeze_encoder_epochs=FREEZE_ENCODER_EPOCHS,
            unfreeze_encoder_lr=UNFREEZE_ENCODER_LR,
            focal_gamma=FOCAL_GAMMA,
            focal_alpha=FOCAL_ALPHA,
            head_dropout=HEAD_DROPOUT,
            freeze_encoder_fully=FREEZE_ENCODER_FULLY,
            ablation_mode=variant,
            matching_projection_dim=MATCHING_PROJECTION_DIM,
            matching_negatives_per_positive=MATCHING_NEGATIVES_PER_POSITIVE,
            matching_loss_lambda=MATCHING_LOSS_LAMBDA,
            matching_triplet_margin=MATCHING_TRIPLET_MARGIN,
            run_tag=run_tag,
        )

        if RUN_EXPORT:
            export_predictions(
                checkpoint_path=checkpoint_path,
                output_dir=export_output_base,
                batch_size=BATCH_SIZE,
                splits=('val', 'test', 'unseen_val', 'unseen_test'),
            )

        if RUN_EXTERNAL_EVAL:
            evaluate_external(
                checkpoint_path=checkpoint_path,
                output_dir=external_output_base,
                batch_size=BATCH_SIZE,
            )

        metrics = {}
        if metrics_path.exists():
            metrics = json.loads(metrics_path.read_text())

        results.append({
            'variant': variant,
            'seed': seed,
            'checkpoint_path': str(checkpoint_path),
            'metrics_path': str(metrics_path),
            'prediction_dir': str(prediction_dir),
            'external_dir': str(external_dir),
            'test_f1': metrics.get('test_f1'),
            'unseen_test_f1': metrics.get('unseen_test_f1'),
            'test_0_64_f1': metrics.get('test_0_64_f1'),
            'unseen_test_0_64_f1': metrics.get('unseen_test_0_64_f1'),
            'optimal_threshold': metrics.get('optimal_threshold'),
            'status': 'completed',
        })

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

results_df = pd.DataFrame(results)
results_df


In [ ]:
from pathlib import Path
import json
import pandas as pd

base_experiment_dir = Path(DRIVE_DATA) / 'experiments' / 'cwe_matching'
predictions_base = Path(DRIVE_DATA) / 'analysis' / 'func_finetune_predictions'
external_base = Path(DRIVE_DATA) / 'analysis' / 'external_eval_tier1'

rows = []
for variant in [v.upper() for v in ABLATION_MODES]:
    variant_lower = variant.lower()
    for seed in SEEDS:
        seed_subdir = f'seed{seed}' if len(SEEDS) > 1 else ''
        variant_dir = base_experiment_dir / variant_lower / seed_subdir if seed_subdir else base_experiment_dir / variant_lower
        pred_dir = (predictions_base / variant_lower / seed_subdir) if seed_subdir else (predictions_base / variant_lower)
        ext_dir = (external_base / variant_lower / seed_subdir) if seed_subdir else (external_base / variant_lower)

        metrics_path = variant_dir / 'func_finetune_metrics.json'
        export_manifest = pred_dir / 'manifest.json'
        external_metrics = ext_dir / 'metrics.json'

        train_metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
        external_summary = json.loads(external_metrics.read_text()) if external_metrics.exists() else {}

        row = {
            'variant': variant,
            'seed': seed,
            'test_f1': train_metrics.get('test_f1'),
            'unseen_test_f1': train_metrics.get('unseen_test_f1'),
            'test_0_64_f1': train_metrics.get('test_0_64_f1'),
            'unseen_test_0_64_f1': train_metrics.get('unseen_test_0_64_f1'),
            'optimal_threshold': train_metrics.get('optimal_threshold'),
            'export_manifest': str(export_manifest) if export_manifest.exists() else None,
            'external_metrics': str(external_metrics) if external_metrics.exists() else None,
        }

        for dataset_name, dataset_metrics in external_summary.items():
            row[f'{dataset_name}_f1'] = dataset_metrics.get('f1')
            row[f'{dataset_name}_retrieval_top1'] = dataset_metrics.get('retrieval_top1')
            row[f'{dataset_name}_retrieval_top3'] = dataset_metrics.get('retrieval_top3')

        rows.append(row)

comparison_df = pd.DataFrame(rows)

# Per-variant aggregation across seeds (mean ± std)
numeric_cols = [c for c in comparison_df.columns if c not in {'variant', 'seed', 'export_manifest', 'external_metrics'}]
agg_df = comparison_df.groupby('variant')[numeric_cols].agg(['mean', 'std']).round(4)

print('Per-run results:')
display(comparison_df)
print('Aggregated across seeds:')
display(agg_df)


## Notes

- `export_predictions()` writes to `data/analysis/func_finetune_predictions/<variant>/...`
- `evaluate_external()` writes to `data/analysis/external_eval_tier1/<variant>/...`
- If Colab runs out of memory during multi-variant execution, reduce `BATCH_SIZE`, reduce `ABLATION_MODES`, or run variants in smaller batches.

- Set `SKIP_COMPLETED_VARIANTS=True` to resume a sweep without retraining finished variants.
- Use `FORCE_RERUN_VARIANTS = ['E3']` to rerun specific variants while still skipping the others.

- Existing checkpoints from before the checkpoint-serialization fix should be deleted or rerun via `FORCE_RERUN_VARIANTS` before export/eval.
